# Chest X-ray Pneumonia Classifier

## Imports

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import kagglehub
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

IMG_SIZE = (180, 180)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

## Download the dataset

In [ ]:
path = Path(kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia"))
train_dirs = [p for p in path.rglob("train") if p.is_dir()]
DATA_DIR = min(train_dirs, key=lambda p: len(p.parts)).parent
print("Data folder:", DATA_DIR)

## Load the data

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "train", image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="binary", validation_split=0.1, subset="training", seed=123)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "train", image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="binary", validation_split=0.1, subset="validation", seed=123)

test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "test", image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="binary", shuffle=False)

class_names = train_ds.class_names   # save before cache/prefetch
print("Classes:", class_names)       # NORMAL = 0, PNEUMONIA = 1

## Preview the images

In [ ]:
plt.figure(figsize=(10, 5))
for images, labels in train_ds.take(1):
    for i in range(8):
        plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i][0])])
        plt.axis("off")
plt.show()

## Prepare the datasets

In [ ]:
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

## Calculate class weights

In [ ]:
counts = [len(list((DATA_DIR / "train" / c).glob("*.jp*g"))) for c in class_names]
total = sum(counts)
class_weight = {i: total / (2 * n) for i, n in enumerate(counts)}
print("Image counts:", dict(zip(class_names, counts)))
print("Class weights:", class_weight)

## Build the model

In [ ]:
model = models.Sequential([
    layers.Input(shape=IMG_SIZE + (3,)),

    # augmentation (inactive at prediction time)
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),

    layers.Rescaling(1.0 / 255),

    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(4),

    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),

    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy",
              metrics=["accuracy"])
model.summary()

## Train the model

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=12,
                    callbacks=callbacks, class_weight=class_weight)

## Plot training progress

In [ ]:
h = history.history
epochs = range(1, len(h["accuracy"]) + 1)

plt.figure(figsize=(12, 4))
for i, (key, title) in enumerate([("accuracy", "Accuracy"), ("loss", "Loss")]):
    plt.subplot(1, 2, i + 1)
    plt.plot(epochs, h[key], marker="o", label="Train")
    plt.plot(epochs, h["val_" + key], marker="o", label="Validation")
    plt.title(title); plt.xlabel("Epoch"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Evaluate the model

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds]).ravel().astype(int)
y_prob = model.predict(test_ds, verbose=0).ravel()
y_pred = (y_prob > 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=class_names))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=class_names, cmap="Blues")
plt.show()

## Try one image

In [ ]:
sample = sorted((DATA_DIR / "test" / "PNEUMONIA").glob("*.jp*g"))[4]
img = tf.keras.utils.load_img(sample, target_size=IMG_SIZE)

x = np.expand_dims(tf.keras.utils.img_to_array(img), axis=0)
score = model.predict(x, verbose=0)[0][0]

plt.imshow(img); plt.axis("off"); plt.show()
print(f"Score: {score:.3f} ->", class_names[int(score > 0.5)])

## Save the model

In [ ]:
model.save("chestscan_cnn.keras")